# Ginger Pest & Disease Classification — MobileNetV2 Transfer Learning

**Dataset:** 9 classes from `ginger_plant_dataset/combined/`  
**Classes:** Damage-Pest, Dehydrated, Healthy, Leaf roller, Leaf-blight,  
           Mite infestation, Rhizome scale, Shoot borer, Thrips  
**Output:** `ginger_pest_model.tflite` — download and place in `streamlit_app/models/`

> **Colab instructions:**
> 1. Upload your dataset folder to Google Drive at `My Drive/Datasets/ginger_plant_dataset/`.
> 2. Run all cells in order (Runtime → Run all).
> **Output:** `ginger_pest_model.tflite` — download and place in `streamlit_app/models/`

## Phase 1 — Environment & Drive Setup

In [ ]:
import os
import sys
import shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # ── Dataset location in Google Drive ──
    DATASET_ROOT = '/content/drive/MyDrive/Datasets/ginger_plant_dataset'
else:
    # Local: look for dataset in project directory
    project_root = os.path.dirname(os.getcwd())  # go up from notebooks/
    DATASET_ROOT = os.path.join(project_root, 'ginger_plant_dataset')

# We use a 'combined' folder inside the dataset root
COMBINED_DIR = os.path.join(DATASET_ROOT, 'combined')

# Check if combined/ has class folders directly (no train/val split)
# Structure: combined/Damage-Pest/, combined/Healthy/, ...
if os.path.exists(COMBINED_DIR):
    # Class folders are directly inside combined/ (no train/ subfolder)
    TRAIN_DIR = COMBINED_DIR
    VAL_DIR = None
    HAS_VAL = False
    print('Found combined/ with class folders directly inside.')
else:
    # Fallback: use the old train/ and validation/ folders
    TRAIN_DIR = os.path.join(DATASET_ROOT, 'train')
    VAL_DIR   = os.path.join(DATASET_ROOT, 'validation')
    HAS_VAL = os.path.exists(VAL_DIR)
    print(f'Using train/ and validation/ folders.')

# If external dataset exists locally, copy it to the project
EXTERNAL_DATASET = r'E:\\01_WORKSPACE\\03_TESSERACT_IOT\\01_PROJECTS\\2026-04-26_Ashok_Sadaware_GingerCrop_Health_Monitor\\01_Inputs\\datasets\\Ginger_Leaf_Dataset\\classes'
if not IN_COLAB and not os.path.exists(COMBINED_DIR) and os.path.exists(EXTERNAL_DATASET):
    print(f'\\nExternal dataset found at: {EXTERNAL_DATASET}')
    print('Copying to project ginger_plant_dataset/combined/ ...')
    shutil.copytree(EXTERNAL_DATASET, COMBINED_DIR, dirs_exist_ok=True)
    TRAIN_DIR = COMBINED_DIR
    HAS_VAL = False
    print(f'Copied {len(os.listdir(COMBINED_DIR))} class folders.')

print(f'\\nDATASET_ROOT : {DATASET_ROOT}')
print(f'TRAIN_DIR    : {TRAIN_DIR}')
print(f'Has validation: {HAS_VAL}  {"(will split from train)" if not HAS_VAL else ""}')

# List classes and count images
img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

if os.path.exists(TRAIN_DIR):
    classes = sorted([c for c in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, c))])
    for cls in classes:
        n = len([f for f in os.listdir(os.path.join(TRAIN_DIR, cls))
                 if os.path.splitext(f)[1].lower() in img_exts])
        print(f'  {cls}: {n} images')
    print(f'  ── Total classes: {len(classes)}')

if HAS_VAL and os.path.exists(VAL_DIR):
    print(f'\\nValidation set:')
    val_classes = sorted([c for c in os.listdir(VAL_DIR) if os.path.isdir(os.path.join(VAL_DIR, c))])
    for cls in val_classes:
        n = len([f for f in os.listdir(os.path.join(VAL_DIR, cls))
                 if os.path.splitext(f)[1].lower() in img_exts])
        print(f'  {cls}: {n} images')

## Phase 2 — Image Parameters & Data Generators

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH  = 224
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 0.001
MODEL_SAVE = 'ginger_pest_model.h5'
CLASS_NAMES = [
    'Damage-Pest',
    'Dehydrated',
    'Healthy',
    'Leaf roller',
    'Leaf-blight',
    'Mite infestation',
    'Rhizome scale',
    'Shoot borer',
    'Thrips',
]
NUM_CLASSES = len(CLASS_NAMES)

print(f'Number of classes: {NUM_CLASSES}')
print(f'Class names: {CLASS_NAMES}')
print(f'Image size : {IMG_HEIGHT}x{IMG_WIDTH}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Max epochs : {EPOCHS}')

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ── Data augmentation for training ──
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
)
val_datagen = ImageDataGenerator(rescale=1.0 / 255)


if HAS_VAL:
    # Separate train/ and validation/ folders exist
    train_gen = train_datagen.flow_from_directory(
        TRAIN_DIR,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=True,
        seed=42,
    )
    val_gen = val_datagen.flow_from_directory(
        VAL_DIR,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False,
    )
else:
    # No validation folder — use 80/20 split from train/
    print('No validation/ folder found. Using 20% of data for validation.')
    train_datagen_split = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.15,
        zoom_range=0.2,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.8, 1.2],
        fill_mode='nearest',
        validation_split=0.2,
    )

    train_gen = train_datagen_split.flow_from_directory(
        TRAIN_DIR,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training',
        shuffle=True,
        seed=42,
    )
    val_gen = train_datagen_split.flow_from_directory(
        TRAIN_DIR,
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation',
        shuffle=False,
        seed=42,
    )

import json
class_indices = train_gen.class_indices
print('\nclass_indices:', class_indices)
with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=2)
print('class_indices.json saved.')
print(f'Training samples  : {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')

## Phase 3 — Build MobileNetV2 Transfer Learning Model

Multi-class output with 9 classes using softmax activation.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
)
base_model.trainable = False  # freeze pretrained weights

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(NUM_CLASSES, activation='softmax')(x)  # multi-class output

model = Model(inputs=base_model.input, outputs=output)
model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## Phase 4 — Train with EarlyStopping & ModelCheckpoint

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        MODEL_SAVE,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
)

print(f'\nBest val_accuracy : {max(history.history["val_accuracy"]):.4f}')

### Phase 4b — Fine-tune the top layers (optional)

Unfreeze the last ~30 layers of MobileNetV2 and continue training with a lower learning rate for better accuracy.

In [ ]:
# ── Fine-tuning phase ──
# Unfreeze the base model and train with a lower LR
base_model.trainable = True

# Freeze early layers, only fine-tune the later ones
fine_tune_at = len(base_model.layers) - 30  # last ~30 layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=LR / 10),  # 10x lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print(f'Fine-tuning from layer {fine_tune_at} / {len(base_model.layers)}')
print(f'Learning rate: {LR / 10}')

fine_tune_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        MODEL_SAVE.replace('.h5', '_finetuned.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

# Continue training from best weights
history_fine = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=fine_tune_callbacks,
)

print(f'\nAfter fine-tuning - Best val_accuracy : {max(history_fine.history["val_accuracy"]):.4f}')

## Phase 5 — Loss / Accuracy Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Combine both training histories if fine-tuning was done
if 'history_fine' in dir() and history_fine is not None:
    all_loss = history.history['loss'] + history_fine.history['loss']
    all_val_loss = history.history['val_loss'] + history_fine.history['val_loss']
    all_acc = history.history['accuracy'] + history_fine.history['accuracy']
    all_val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
    # Add a split marker
    split_epoch = len(history.history['loss'])
else:
    all_loss = history.history['loss']
    all_val_loss = history.history['val_loss']
    all_acc = history.history['accuracy']
    all_val_acc = history.history['val_accuracy']
    split_epoch = None

epochs_ran = range(1, len(all_loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(epochs_ran, all_loss,     label='Train Loss', linewidth=2)
axes[0].plot(epochs_ran, all_val_loss, label='Val Loss', linewidth=2)
if split_epoch:
    axes[0].axvline(x=split_epoch, color='gray', linestyle='--', alpha=0.5, label='Fine-tune start')
axes[0].set_title('Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, all_acc,     label='Train Accuracy', linewidth=2)
axes[1].plot(epochs_ran, all_val_acc, label='Val Accuracy', linewidth=2)
if split_epoch:
    axes[1].axvline(x=split_epoch, color='gray', linestyle='--', alpha=0.5, label='Fine-tune start')
axes[1].set_title('Accuracy', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ginger_training_curves.png', dpi=150)
plt.show()
print('Saved ginger_training_curves.png')

## Phase 6 — Confusion Matrix & Classification Report

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

val_gen.reset()
y_pred_prob = model.predict(val_gen, verbose=1)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = val_gen.classes

idx_to_class = {v: k for k, v in class_indices.items()}
class_names  = [idx_to_class[i] for i in sorted(idx_to_class)]

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(12, 10))
disp.plot(cmap='Blues', ax=ax, xticks_rotation=45)
plt.title('Confusion Matrix — Validation Set', fontsize=14)
plt.tight_layout()
plt.savefig('ginger_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

## Phase 7 — Sanity Test (one image from each class)

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

def predict_one(img_path, model, class_indices):
    img = keras_image.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    arr = keras_image.img_to_array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)
    probs = model.predict(arr, verbose=0)[0]  # array of 9 probabilities
    pred_idx = int(np.argmax(probs))
    confidence = float(probs[pred_idx])
    idx_to_class = {v: k for k, v in class_indices.items()}
    label = idx_to_class[pred_idx]
    return label, confidence, img

for cls_name in sorted(class_indices.keys()):
    cls_dir = os.path.join(TRAIN_DIR, cls_name)
    if not os.path.exists(cls_dir):
        continue
    # Find first image file (skip .json etc.)
    img_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
    samples = [f for f in os.listdir(cls_dir)
               if os.path.splitext(f)[1].lower() in img_exts]
    if not samples:
        continue
    sample = samples[0]
    img_path = os.path.join(cls_dir, sample)
    label, conf, pil_img = predict_one(img_path, model, class_indices)
    print(f'True: {cls_name:20s}  →  Predicted: {label:20s}  (confidence {conf*100:.1f}%)')
    plt.figure(figsize=(3, 3))
    plt.imshow(pil_img)
    plt.title(f'True: {cls_name}\nPred: {label} ({conf*100:.1f}%)', fontsize=9)
    plt.axis('off')
    plt.show()

## Phase 8 — Download Model

After training, place `ginger_pest_model.tflite` inside `streamlit_app/models/`.

In [ ]:
# Find the best model file
best_model = MODEL_SAVE
finetuned_model = MODEL_SAVE.replace('.h5', '_finetuned.h5')

if os.path.exists(finetuned_model):
    best_model = finetuned_model
    print(f'Using fine-tuned model: {best_model}')
else:
    print(f'Using base model: {best_model}')

model_size_mb = os.path.getsize(best_model) / (1024 * 1024)
print(f'Model file size: {model_size_mb:.2f} MB')
assert model_size_mb > 1, 'Model file is suspiciously small — check training output!'

if IN_COLAB:
    from google.colab import files
    files.download(best_model)
    files.download('class_indices.json')
    files.download('ginger_training_curves.png')
    files.download('ginger_confusion_matrix.png')
else:
    print(f'Local run — model saved to: {os.path.abspath(best_model)}')

## Phase 9 — Convert to TFLite

Converts the trained Keras model to TFLite format.  
Two options:
- **Float32** (default) — best accuracy, slightly larger file
- **INT8 quantization** — smaller file, faster inference, suitable for edge/embedded devices

Download and place the `.tflite` file in `streamlit_app/models/`.

In [ ]:
import os
import numpy as np
import tensorflow as tf

H5_SAVE     = best_model  # use the best model from training
TFLITE_SAVE = "ginger_pest_model.tflite"
TFLITE_INT8 = "ginger_pest_model_int8.tflite"
IMG_HEIGHT, IMG_WIDTH = 224, 224

# Load from .h5 if model is not already in memory
if "model" not in dir():
    print(f'Loading {H5_SAVE} from disk...')
    model = tf.keras.models.load_model(H5_SAVE)
    print('Model loaded.')

# ── Option 1: Float32 TFLite (no quantization) ──
print('\n--- Converting to Float32 TFLite ---')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()

with open(TFLITE_SAVE, 'wb') as f:
    f.write(tflite_bytes)

tflite_mb = os.path.getsize(TFLITE_SAVE) / (1024 * 1024)
print(f'Float32 TFLite: {TFLITE_SAVE}  ({tflite_mb:.2f} MB)')
assert tflite_mb > 0.5, 'TFLite file suspiciously small — check model'

# ── Option 2: INT8 Quantized TFLite (smaller, faster) ──
print('\n--- Converting to INT8 Quantized TFLite ---')

# Representative dataset for quantization calibration
def representative_dataset():
    # Use a subset of validation images for calibration
    val_gen.reset()
    for _ in range(100):  # 100 batches of calibration data
        batch = val_gen.next()
        yield [batch[0].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

try:
    tflite_int8_bytes = converter.convert()
    with open(TFLITE_INT8, 'wb') as f:
        f.write(tflite_int8_bytes)
    int8_mb = os.path.getsize(TFLITE_INT8) / (1024 * 1024)
    print(f'INT8 TFLite: {TFLITE_INT8}  ({int8_mb:.2f} MB)')
    print(f'  Compression ratio: {tflite_mb / int8_mb:.1f}x smaller')
except Exception as e:
    print(f'INT8 quantization failed: {e}')
    print('Skipping INT8 — float32 model is still available.')

# ── Sanity check: run dummy inference through float32 TFLite ──
print('\n--- Sanity Check ---')
tflite_interp = tf.lite.Interpreter(model_path=TFLITE_SAVE)
tflite_interp.allocate_tensors()
inp_det = tflite_interp.get_input_details()
out_det = tflite_interp.get_output_details()

print(f'Input shape:  {inp_det[0]["shape"]}')
print(f'Output shape: {out_det[0]["shape"]}  (should be [1, 9])')
assert out_det[0]['shape'][1] == NUM_CLASSES, \
    f'Output layer has {out_det[0]["shape"][1]} classes, expected {NUM_CLASSES}'

dummy = np.zeros((1, IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.float32)
tflite_interp.set_tensor(inp_det[0]['index'], dummy)
tflite_interp.invoke()
tflite_out = tflite_interp.get_tensor(out_det[0]['index'])[0]
print(f'Dummy inference output shape: {tflite_out.shape}')
print(f'Class probabilities: {np.round(tflite_out, 3)}')
print(f'Sum of probabilities: {np.sum(tflite_out):.4f}  (should be ~1.0)')
print('\nTFLite sanity check passed!')

if IN_COLAB:
    from google.colab import files
    files.download(TFLITE_SAVE)
    if os.path.exists(TFLITE_INT8):
        files.download(TFLITE_INT8)
    print(f'\nDownloaded — place in streamlit_app/models/ folder')
else:
    print(f'\nLocal run — TFLite models saved to:')
    print(f'  {os.path.abspath(TFLITE_SAVE)}')
    if os.path.exists(TFLITE_INT8):
        print(f'  {os.path.abspath(TFLITE_INT8)}')
    print(f'\n➡ Copy to streamlit_app/models/ for use in the app.')

## Phase 10 — Copy TFLite to Streamlit App

Run this cell to automatically copy the TFLite model into the Streamlit app's models folder.

In [ ]:
if not IN_COLAB:
    import shutil
    
    # Detect the streamlit app models directory
    possible_paths = [
        os.path.join(project_root, 'streamlit_app', 'models'),
        os.path.join(os.getcwd(), '..', 'streamlit_app', 'models'),
        os.path.join(os.getcwd(), 'streamlit_app', 'models'),
    ]
    
    app_models_dir = None
    for p in possible_paths:
        resolved = os.path.abspath(p)
        if os.path.exists(resolved):
            app_models_dir = resolved
            break
    
    if app_models_dir:
        # Copy float32 model
        dst = os.path.join(app_models_dir, TFLITE_SAVE)
        shutil.copy2(TFLITE_SAVE, dst)
        print(f'✓ Copied {TFLITE_SAVE} → {dst}')
        
        # Copy INT8 model if it exists
        if os.path.exists(TFLITE_INT8):
            dst_int8 = os.path.join(app_models_dir, TFLITE_INT8)
            shutil.copy2(TFLITE_INT8, dst_int8)
            print(f'✓ Copied {TFLITE_INT8} → {dst_int8}')
        
        # Copy class indices
        dst_idx = os.path.join(app_models_dir, 'class_indices.json')
        shutil.copy2('class_indices.json', dst_idx)
        print(f'✓ Copied class_indices.json → {dst_idx}')
    else:
        print('⚠ Could not find streamlit_app/models/ directory.')
        print(f'  Manually copy these files to streamlit_app/models/:')
        print(f'    - {os.path.abspath(TFLITE_SAVE)}')
        if os.path.exists(TFLITE_INT8):
            print(f'    - {os.path.abspath(TFLITE_INT8)}')
        print(f'    - {os.path.abspath("class_indices.json")}')
else:
    print('Colab run — download the files above and manually place in streamlit_app/models/')